# Watergraafsmeer 

Notebook bevat code:
* Voor inlezen van data van parkeerplaatsen
* Inlezen van data op gemeenten
* Filteren op Watergraafsmeer
* Bereken
    * Benodigde aantal drinkwaterpunten
    * Benodigde aantal parkeerplaatsen
* Wijs parkeerplaatsen toe aan bewoners

**Table of contents**<a id='toc0_'></a>    
- [Watergraafsmeer](#toc1_1_1_)    
      - [First of all, make sure to adjust the BASE_FOLDER_LOCATION under "eda_support_files/CONSTANTS.py"](#toc1_1_1_1_)    
    - [Collecting and filtering the OpenStreetMap dataset for parking lots](#toc1_1_2_)    
      - [For the case of 'nooddrinkwater' the eligible locations are parking lots, and OpenStreetMap is used to find these parking lots](#toc1_1_2_1_)    
      - [Filtering the OSM parking lots to certain types (e.g. minimal size of parking lot)](#toc1_1_2_2_)    
    - [Collecting and processing Gemeenten/wijken/buurten data (incl. geometry) via PDOK](#toc1_1_3_)    
      - [Gemeente/wijk/buurten data is necessary for:](#toc1_1_3_1_)    
    - [Filter on Watergraafsmeer](#toc1_1_4_)    
    - [Calculate the necessary amounts of drinkwaterpunten for Watergraafsmeer](#toc1_1_5_)    
    - [Select parking lots within Watergraafsmeer, sum how many in each gemeente and whether that is enough.](#toc1_1_6_)    
    - [Let's add the residents](#toc1_1_7_)    
    - [For each gemeente decide parking lots and resident assignment to parking lots](#toc1_1_8_)    
    - [Use the code below to get info about experiments](#toc1_1_9_)    
    - [Summary of evaluation metrics](#toc1_1_10_)    
  - [Export to GPKG](#toc1_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

> Make sure to adjust the BASE_FOLDER_LOCATION under "eda_support_files/CONSTANTS.py"

In [ ]:
%pip install osmnx contextily folium ipyleaflet cbsodata ydata-sdk ortools scikit-learn plotly matplotlib ipyleaflet

In [ ]:
GEMEENTE = "Amsterdam"
MINIMUM_PARKING_SIZE = 1
MAXIMUM_PEOPLE = 50

In [ ]:
import logging
from logging.handlers import RotatingFileHandler
from pathlib import Path
from typing import Optional, Union, Optional
from IPython.display import display, HTML
from eda_support_files.CONSTANTS import DATA_EXTERNAL_FOLDER_LOCATION, DATA_INTERIM_FOLDER_LOCATION, DATA_PROCESSED_FOLDER_LOCATION, DATA_RAW_FOLDER_LOCATION
import numpy as np
import geopandas as gpd
from eda_support_files.GetOSMData import GetOSMData
from eda_support_files.FilterOSMData import FilterOSMData
import pandas as pd
from pathlib import Path
from eda_support_files.GetGemeenteDataPDOK import GetGemeenteDataPDOK
from eda_support_files.GenerateGemeenteResidents import GenerateGemeenteResidents
from eda_support_files.create_visualisation import create_interactive_map
import matplotlib.colors as mcolors
from matplotlib import colormaps
import numpy as np
import random
from eda_support_files.modelling.Orchestrator import ExperimentOrchestrator
import os

In [ ]:
# Add all necessary variables
ASSIGNED_RESIDENT_FOLDER_LOCATION = f"{DATA_INTERIM_FOLDER_LOCATION}/watergraafsmeer_residents_assigned"
STATLINE_DIR = Path(f"{DATA_EXTERNAL_FOLDER_LOCATION}/statline_85618NED")
RUN_VISUALISATIONS = True

logging.getLogger("py4j").disabled = True

### <a id='toc1_1_2_'></a>[Collecting and filtering the OpenStreetMap dataset for parking lots](#toc0_)

#### <a id='toc1_1_2_1_'></a>[For the case of 'nooddrinkwater' the eligible locations are parking lots, and OpenStreetMap is used to find these parking lots](#toc0_)

In [ ]:
osm_data_getter = GetOSMData()
gdf_all: gpd.GeoDataFrame = osm_data_getter.run(
    folder=DATA_EXTERNAL_FOLDER_LOCATION,
    only_load=True
)

# Area of each geometry in square meters
if "area_m2" not in gdf_all.columns:
    gdf_all["area_m2"] = gdf_all.geometry.area.round(4)

# Perimeter / boundary length in meters
if "perimeter_m" not in gdf_all.columns:
    gdf_all["perimeter_m"] = gdf_all.geometry.length

# Polsby–Popper compactness measure "how square a parking lot is", to prevent long and small lots.
# (4πA) / P² where A = area, P = perimeter
if "compactness" not in gdf_all.columns:
    gdf_all["compactness"] = (
        4 * np.pi * gdf_all["area_m2"]
    ) / (gdf_all["perimeter_m"] ** 2)


#### <a id='toc1_1_2_2_'></a>[Filtering the OSM parking lots to certain types (e.g. minimal size of parking lot)](#toc0_)

In [ ]:
osm_data_filterer = FilterOSMData(
        allowed_parking_types=None,
        min_parking_size=MINIMUM_PARKING_SIZE,
        expected_crs_meters="EPSG:28992"
)
gdf_parking_lots = osm_data_filterer.run(gdf_all, folder=DATA_INTERIM_FOLDER_LOCATION, only_load=False)

# Fix crs for everything else after calculating area
gdf_parking_lots = gdf_parking_lots.to_crs(epsg=4326)

### <a id='toc1_1_3_'></a>[Collecting and processing Gemeenten/wijken/buurten data (incl. geometry) via PDOK](#toc0_)

#### <a id='toc1_1_3_1_'></a>[Gemeente/wijk/buurten data is necessary for:](#toc0_)
- the municipal boundaries
- data on number of residents (which determines the number of nooddrinkwaterpunten)  

In [ ]:
gemeente_data_getter = GetGemeenteDataPDOK(filepath=DATA_RAW_FOLDER_LOCATION)
gdf_gemeenten_buurten = gemeente_data_getter.run(load_from_file=True)

display(gdf_gemeenten_buurten.sample(5))

In [ ]:
# --- CONFIG ---
left_df = gdf_gemeenten_buurten.copy()

# --- 1) Load Observations ---
obs = pd.read_csv(STATLINE_DIR / "Observations.csv",
                  sep=";", encoding="utf-8-sig", dtype=str)
obs = obs[["WijkenEnBuurten", "Measure", "Value"]].copy()
obs["WijkenEnBuurten"] = obs["WijkenEnBuurten"].str.strip()
obs["Measure"] = obs["Measure"].str.strip()

# --- 1.5) Value's can have a ',' which is a problem for pandas, so lets replace them with '.'
obs["Value"] = obs["Value"].str.replace(",", ".")
obs["Value"] = pd.to_numeric(obs["Value"], errors="coerce")

# Drop duplicates (region + measure)
obs = obs.drop_duplicates(subset=["WijkenEnBuurten", "Measure"])

# --- 2) Pivot to wide ---
wide = (
    obs.pivot_table(
        index="WijkenEnBuurten",
        columns="Measure",
        values="Value",
        aggfunc="first"
    )
    .reset_index()
)

# --- 3) Load MeasureCodes and MeasureGroups ---
measures = pd.read_csv(STATLINE_DIR / "MeasureCodes.csv",
                       sep=";", encoding="utf-8-sig", dtype=str)[["Identifier", "Title", "MeasureGroupId"]]
groups = pd.read_csv(STATLINE_DIR / "MeasureGroups.csv",
                     sep=";", encoding="utf-8-sig", dtype=str)[["Id", "Title", "ParentId"]]
groups = groups.rename(columns={"Id": "MeasureGroupId", "Title": "GroupTitle"})

# Build lookups
group_lookup = dict(zip(groups["MeasureGroupId"], groups["GroupTitle"]))
parent_lookup = dict(zip(groups["MeasureGroupId"], groups["ParentId"]))
parent_title_lookup = dict(zip(groups["MeasureGroupId"], groups["GroupTitle"]))

# --- 4) Build combined label: Parent - Group - Measure ---
def build_full_label(row):
    group_id = row["MeasureGroupId"]
    group_title = group_lookup.get(group_id, "")
    parent_id = parent_lookup.get(group_id)
    parent_title = parent_title_lookup.get(parent_id, "") if pd.notna(parent_id) else ""
    parts = [p for p in [parent_title, group_title, row["Title"]] if p]
    return " - ".join(parts)

label_map = {row["Identifier"]: build_full_label(row) for _, row in measures.iterrows()}

# Rename columns in wide DataFrame
wide = wide.rename(columns=label_map)


# --- 5) Merge with your left_df ---
gdf_gemeenten_buurten_dem = left_df.merge(wide, left_on="buurtcode", right_on="WijkenEnBuurten", how="left")
gdf_gemeenten_buurten_dem = gdf_gemeenten_buurten_dem[[
    "WijkenEnBuurten","geometry", "buurtnaam", "buurtcode", "gemeentenaam", "aantal_inwoners", "level"
]]

display(gdf_gemeenten_buurten_dem.sample(5))
gdf_gemeenten_buurten_dem.to_csv(f"{DATA_PROCESSED_FOLDER_LOCATION}/combined_dataset.csv", index=False)

### <a id='toc1_1_4_'></a>[Filter on Watergraafsmeer](#toc0_)

In [ ]:
watergraafsmeer = [
    "Drieburg",
    "Nieuwe Oosterbegraafplaats",
    "Betondorp",
    "Park de Meer",
    "Sportpark Middenmeer-Zuid",
    "Science Park-Zuid",
    "Sportpark Middenmeer-Noord",
    "De Wetbuurt",
    "Tuindorp Frankendael",
    "Middenmeer-Zuid",
    "Science Park-Noord",
    "Middenmeer-Noord",
    "Linnaeusparkbuurt",
    "Frankendael",
    "Don Bosco",
    "De Eenhoorn",
    "Julianapark",
    "Tuindorp Amstelstation",
    "Sportpark Voorland"
]

omgeving = [
    "Weespertrekvaart",
    "Amstelkwartier-Zuid",
    "Amstelkwartier-Noord",
    "Rijnbuurt-Oost",
    "Kromme Mijdrechtbuurt",
    "IJselbuurt-Oost",
    "De Omval",
    "Van der Kunbuurt",
    "De Eenhoorn",
    "Transvaalbuurt-West",
    "Parooldriehoek",
    "Transvaalbuurt-Oost",
    "Oostpoort",
    "Ambonpleinbuurt",
    "Sumatraplantsoenbuurt",
    "Flevopark",
    "Oosterparkbuurt-Zuidwest",
    "Oosterparkbuurt-Zuidoost",
    "Dapperbuurt-Zuid",
    "Timorpleinbuurt-Zuid",
    "Makassarpleinbuurt",
    "Zeeburgerdijk-Oost"
]

all_buurten = watergraafsmeer + omgeving

gdf_watergraafsmeer = gdf_gemeenten_buurten_dem[
    gdf_gemeenten_buurten_dem["buurtnaam"].isin(all_buurten) &
    (gdf_gemeenten_buurten_dem["gemeentenaam"] == "Amsterdam")
].copy()


In [ ]:
gdf_watergraafsmeer.head()

### <a id='toc1_1_5_'></a>[Calculate the necessary amounts of drinkwaterpunten for Watergraafsmeer](#toc0_)

In [ ]:
from eda_support_files.CalcBenodigdWaterpunt import CalcBenodigdWaterpunt

calc_benodigd_waterpunt_procesessor = CalcBenodigdWaterpunt(max_citizens_per_point = MAXIMUM_PEOPLE)
gdf_watergraafsmeer = calc_benodigd_waterpunt_procesessor.run(gdf_watergraafsmeer)

gdf_watergraafsmeer


### <a id='toc1_1_6_'></a>[Select parking lots within Watergraafsmeer and whether that is enough.](#toc0_)

In [ ]:
from eda_support_files.CombineGemeenteAndParkinglotData import CombineGemeenteAndParkinglotData

gemeente_parking_combiner = CombineGemeenteAndParkinglotData()
gdf_parking_lots, gdf_watergraafsmeer = gemeente_parking_combiner.run(
    gdf_parking_lots=gdf_parking_lots, 
    gdf_gemeenten=gdf_watergraafsmeer)

display(gdf_parking_lots[gdf_parking_lots['gemeentenaam'] == GEMEENTE].sample(5))

In [ ]:
from eda_support_files.CombineGemeenteAndParkinglotData import CombineGemeenteAndParkinglotData

gemeente_parking_combiner = CombineGemeenteAndParkinglotData()
gdf_parking_lots, gdf_gemeenten = gemeente_parking_combiner.run(
    gdf_parking_lots=gdf_parking_lots, 
    gdf_gemeenten=gdf_watergraafsmeer)

display(gdf_parking_lots.sample(5))

### <a id='toc1_1_7_'></a>[Let's add the residents](#toc0_)
To be able to calculate distances for residents to the nooddrinkwaterpunt, we need information on where these residents live.
We do not have this data available, but we use the BAG to find out which buildings (verblijfsobjecten) are for residential use (woonfunctie), and divide the residents over the residential verblijfsobjecten 

In [ ]:
residents_folder = f"{DATA_INTERIM_FOLDER_LOCATION}/watergraafsmeer_residents"

gemeente_residents_generator = GenerateGemeenteResidents(residents_folder = residents_folder)
# the generator creates and saves a file with resident locations
result_text = gemeente_residents_generator.run(
    gdf_gemeenten=gdf_gemeenten[gdf_gemeenten['gemeentenaam'] == "Amsterdam"][['gemeentenaam']], 
    gdf_buurten=gdf_watergraafsmeer, 
    overwrite=True
)

### <a id='toc1_1_8_'></a>[For each gemeente decide parking lots and resident assignment to parking lots](#toc0_)

Experimenten:
- MinAvgDistanceSelector: minimize average distance from residents to parking lot
- MinMaxDistanceSelector: minimize maximum distance from residents to parking lot

Assignment:
The above described experiments only determine which parking lots to choose, hich resident has to go to which parking lot is not yet determined and is now determined with 'min_cost_flow', SimpleMinCostFlow from ORtools.

In [ ]:


input_folder = f"{DATA_INTERIM_FOLDER_LOCATION}/watergraafsmeer_residents"

# Run for gemeenten in list
available_gemeente_file_paths = [f"{input_folder}/{gemeentennaam.replace(' ', '_')}.geojson" for gemeentennaam in list(["Amsterdam"])]

SETUP_EXPERIMENTS = {
    "minimum_avg_distance_min_cost_flow": {"optimisation_class": "MinAvgDistanceSelector", "assignment_method": "min_cost_flow"},
    "minimum_max_distance_min_cost_flow": {"optimisation_class": "MinMaxDistanceSelector", "assignment_method": "min_cost_flow"},
}

# Instantiate and run orchestrator
orchestrator = ExperimentOrchestrator(
    gemeente_filepaths = available_gemeente_file_paths,
    setup_experiments = SETUP_EXPERIMENTS,
    output_folder = ASSIGNED_RESIDENT_FOLDER_LOCATION,
    gdf_parking_lots = gdf_parking_lots,
    gdf_gemeenten = gdf_watergraafsmeer
)
orchestrator.run()


### <a id='toc1_1_9_'></a>[Use the code below to get info about experiments](#toc0_)

In [ ]:

# Inputs
experiment_folder = os.path.join(DATA_INTERIM_FOLDER_LOCATION, "watergraafsmeer_residents_assigned", "Amsterdam")

# Discover all .geojson files and load them
def load_experiments(folder: str, experiment_method: str = None) -> dict[str, gpd.GeoDataFrame]:
    """
    Loads all GeoJSON experiment files from the given folder.
    Returns a dict: {experiment_name: GeoDataFrame}
    """
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Experiment folder does not exist: {folder}")

    experiments = {}
    for fname in os.listdir(folder):
        # accept .geojson or .json, case-insensitive
        if fname.lower().endswith((".geojson", ".json")):
            experiment_name = os.path.splitext(fname)[0]  # strip extension
            # Skip if not experiment_method when given
            if experiment_method:
                if experiment_name != experiment_method:
                    continue
            fpath = os.path.join(folder, fname)
            try:
                gdf = gpd.read_file(fpath)
                experiments[experiment_name] = gdf
            except Exception as e:
                # Log and continue loading others
                print(f"[WARN] Failed to read '{fpath}': {e}")

    if not experiments:
        print(f"[INFO] No experiment files found in: {folder}")

    return experiments

# Load all experiments
experiments = load_experiments(experiment_folder)

In [ ]:
gdf_parking_lots_gem = gdf_parking_lots[gdf_parking_lots['gemeentenaam'] == "Amsterdam"]
gdf_gemeenten_gem = gdf_gemeenten[gdf_gemeenten['gemeentenaam'] == "Amsterdam"]

display(HTML(f"<h4>{"Amsterdam"} has {len(experiments.get("minimum_avg_distance_min_cost_flow"))} residents</h2>"))
display(HTML(f"<h4>{"Amsterdam"} has {gdf_gemeenten_gem["aantal_parkeerplaatsen"].iloc[0]} eligible parking lots for nooddrinkwaterpunten and needs {gdf_gemeenten_gem["Benodigd_ceiling"].iloc[0]}</h2>"))

### <a id='toc1_1_10_'></a>[Summary of evaluation metrics](#toc0_)

In [ ]:
from eda_support_files.summarize_results import summarize_assignments, visualize_distances, get_np_bins
from IPython.display import display, HTML
import plotly.graph_objects as go
import pandas as pd

# --- Build global bins across ALL experiments so histograms align
all_distances = pd.concat(
    [gdf["distance_to_parking"] for gdf in experiments.values()],
    ignore_index=True
)

# If get_np_bins expects two arrays, we can pass the same twice to derive bins from the whole set.
# Otherwise, if it accepts an iterable, change accordingly.
np_bins = get_np_bins(all_distances, all_distances)

# --- Create figure and plot each experiment
fig = go.Figure()

all_max_y = []
for exp_name, gdf in sorted(experiments.items()):
    print(exp_name)
    display(HTML(f"<h3>Experiment: {exp_name}</h3>"))
    # Summary (table/metrics)
    summarize_assignments(
        gdf_res=gdf, 
        distances=gdf["distance_to_parking"])

    # Add histogram/trace to figure
    fig, max_y = visualize_distances(
        fig=fig,
        distances=gdf["distance_to_parking"],
        name=exp_name,
        np_bins=np_bins,
        color=None
    )
    all_max_y.append(max_y)
# Add the line for loopafstand
fig.add_shape(type="line", x0=1000, x1=1000, y0=0, y1=max(all_max_y), opacity=1,
                line=dict(color="black", width=4))
# --- Final layout for the combined plot
fig.update_layout(
    title="Distribution of resident-to-parking-lot distances (all experiments)",
    xaxis_title="Distance (m)",
    showlegend=True,
    bargap=0.1
)

fig.show()


## <a id='toc1_2_'></a>[Export to GPKG](#toc0_)

In [ ]:
experiment_method_to_use = "minimum_avg_distance_min_cost_flow"

residents_nh = []
parking_lots_nh = []

experiment_folder = os.path.join(DATA_INTERIM_FOLDER_LOCATION, "watergraafsmeer_residents_assigned", "Amsterdam")
experiments = load_experiments(experiment_folder, experiment_method=experiment_method_to_use)

# === Get residents data ===
gdf_res_plot = experiments.get(experiment_method_to_use).copy()

# === Prepare color mapping ===
unique_lots = gdf_res_plot['assigned_parking_lot'].dropna().unique()
random.shuffle(unique_lots)

# Get colormap and sample colors
cmap = colormaps.get_cmap('gist_rainbow')
colors = [mcolors.to_hex(cmap(x)) for x in np.linspace(0, 1, len(unique_lots))]

# Build mapping: lot ID → color
lot_to_color = {int(lot): color for lot, color in zip(unique_lots, colors)}

# Add color column to residents
gdf_res_plot['color'] = gdf_res_plot['assigned_parking_lot'].map(lot_to_color)

# === Prepare parking lots ===
gdf_parking_lots_gem = gdf_parking_lots[gdf_parking_lots['gemeentenaam'] == "Amsterdam"]
gdf_parking_lots_gem_selected = gdf_parking_lots_gem[
    gdf_parking_lots_gem.index.isin(unique_lots)
][['geometry']].copy()
gdf_parking_lots_gem_selected['color'] = gdf_parking_lots_gem_selected.index.map(lot_to_color)

## Build GeoJSON features for parking lots
parking_features = []
for idx, row in gdf_parking_lots_gem_selected.iterrows():
    props = {"color": row["color"], "type": "parking_lot"}
    parking_features.append({
        "type": "Feature",
        "geometry": row.geometry.__geo_interface__,
        "properties": props
    })

# === Config ===
out_dir = f"{DATA_PROCESSED_FOLDER_LOCATION}/qgis_output/{experiment_method_to_use}/"
gpkg_path = os.path.join(out_dir, "residents_parking_interview.gpkg")

os.makedirs(out_dir, exist_ok=True)

# === Copies of original data ===
residents_copy = gdf_res_plot.copy()
parking_copy = gdf_parking_lots_gem_selected.copy()

# Ensure WGS84 CRS
def to_wgs84(gdf):
    return (gdf.set_crs(4326) if gdf.crs is None else gdf.to_crs(4326))

residents_copy = to_wgs84(residents_copy)
parking_copy = to_wgs84(parking_copy)
parking_copy = parking_copy.reset_index().rename(columns={"index": "parking_lot_id"})

# Prepare columns for export
residents_copy = residents_copy[["assigned_parking_lot", "color", "geometry"]]
parking_copy = parking_copy[["parking_lot_id", "color", "geometry"]]
# combine
residents_nh.append(residents_copy)
parking_lots_nh.append(parking_copy)

# combine all in list
residents_nh_comb = pd.concat(residents_nh).reset_index(drop=True)
parking_lots_nh_comb = pd.concat(parking_lots_nh).reset_index(drop=True)

# === Export GeoPackage ===
residents_nh_comb.to_file(gpkg_path, layer="residents", driver="GPKG")
parking_lots_nh_comb.to_file(gpkg_path, layer="parking_lots", driver="GPKG")